# Setup SageMaker FeatureStore

In [3]:
import boto3
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [4]:
from sagemaker.session import Session

region = boto3.Session().region_name

boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime)

In [5]:
# You can modify the following to use a bucket of your choosing
default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "ndbc-featurestore"

print(default_s3_bucket_name)

sagemaker-us-east-1-318401170150


In [6]:
from sagemaker import get_execution_role

# You can modify the following to use a role of your choosing. See the documentation for how to create this.
role = get_execution_role()
print(role)

arn:aws:iam::318401170150:role/LabRole


In [7]:
# Inspect Dataset
bucket

'sagemaker-us-east-1-318401170150'

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

s3_client = boto3.client("s3", region_name=region)

bucket_name = f"{bucket}"
file_key = (
    "curated/ndbc/buoy=46086/stdmet.parquet"
)
transaction_file_key = (
    "datasets/tabular/fraud_detection/synthethic_fraud_detection_SA/sampled_transactions.csv"
)

data_object = s3_client.get_object(
    Bucket=bucket_name, Key=file_key
)

all_data = pd.read_parquet(io.BytesIO(data_object["Body"].read()))

In [9]:
all_data

,timestamp,station_id,wind_direction,wind_speed,wind_gust,wave_height,dominant_wave_period,average_wave_period,mean_wave_direction,pressure,air_temperature,water_temperature,dewpoint_temperature,wind_speed_ms,wave_energy
0,2023-01-01 00:10:00,46086,172.0,4.7,5.9,1.63,11.43,9.33,283.0,1014.2,14.9,15.9,13.8,2.417887,30.368367
1,2023-01-01 00:40:00,46086,162.0,5.2,6.6,1.88,13.79,9.55,257.0,1014.0,14.9,15.9,14.0,2.675109,48.739376
2,2023-01-01 01:10:00,46086,154.0,5.8,7.2,1.68,12.12,8.98,271.0,1013.5,14.9,15.9,14.1,2.983775,34.207488
3,2023-01-01 01:40:00,46086,162.0,6.2,7.3,1.80,12.12,8.99,270.0,1013.1,15.0,15.9,14.3,3.189553,39.268800
4,2023-01-01 02:10:00,46086,176.0,7.0,8.2,1.76,11.43,8.94,263.0,1012.7,15.0,15.9,14.3,3.601108,35.405568
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35586,2026-01-31 21:20:00,46086,10.0,3.0,4.0,1.30,12.00,9.60,289.0,1017.2,18.2,18.1,16.3,1.543332,20.280000
35587,2026-01-31 21:50:00,46086,10.0,4.0,5.0,1.40,13.00,9.90,282.0,1017.0,18.4,18.0,14.8,2.057776,25.480000
35588,2026-01-31 22:20:00,46086,360.0,4.0,4.0,1.40,13.00,10.10,284.0,1016.8,18.3,17.9,15.8,2.057776,25.480000
35589,2026-01-31 22:50:00,46086,340.0,3.0,4.0,1.30,13.00,10.00,283.0,1016.8,18.4,17.9,15.6,1.543332,21.970000


In [10]:
import sys
from pathlib import Path

# Add the src folder (relative to the notebook) to sys.path
src_path = Path("../src").resolve()  # notebooks/etl_pipeline.ipynb
sys.path.append(str(src_path))

from feature_engineering import make_supervised, time_split

In [34]:
# create supervised features

df_features, feature_cols = make_supervised(
    all_data,
    lead_hours=1,
    lags=[1, 2, 3, 6]
)

In [35]:
# Add feature store metadata columns
df_features["event_time"] = (
    pd.to_datetime(df_features["timestamp"], utc=True)
      .dt.strftime("%Y-%m-%dT%H:%M:%SZ")
)
df_features["record_id"] = (
    df_features["station_id"].astype(str)
    + "_"
    + df_features["event_time"].astype(str)
)
df_features = df_features.drop(columns=["timestamp"])

In [36]:
train_df, val_df, test_df = time_split(df_features)

In [37]:
df_features.head()

,station_id,wind_direction,wind_speed,wind_gust,wave_height,dominant_wave_period,average_wave_period,mean_wave_direction,pressure,air_temperature,...,wave_height_lag_3,E_star_lag_3,wind_speed_lag_3,dominant_wave_period_lag_3,wave_height_lag_6,E_star_lag_6,wind_speed_lag_6,dominant_wave_period_lag_6,event_time,record_id
0,46086,166.0,8.2,10.0,1.91,11.43,8.31,264.0,1011.5,15.2,...,1.80,3.2400,6.2,12.12,1.63,2.6569,4.7,11.43,2023-01-01T03:10:00Z,46086_2023-01-01T03:10:00Z
1,46086,169.0,7.9,10.1,2.04,12.12,7.92,276.0,1011.8,15.3,...,1.76,3.0976,7.0,11.43,1.88,3.5344,5.2,13.79,2023-01-01T03:40:00Z,46086_2023-01-01T03:40:00Z
2,46086,180.0,7.7,9.4,1.96,13.79,7.63,246.0,1011.5,15.2,...,1.86,3.4596,8.5,11.43,1.68,2.8224,5.8,12.12,2023-01-01T04:10:00Z,46086_2023-01-01T04:10:00Z
3,46086,186.0,7.5,9.1,1.80,13.79,7.36,212.0,1011.4,15.4,...,1.91,3.6481,8.2,11.43,1.80,3.2400,6.2,12.12,2023-01-01T04:40:00Z,46086_2023-01-01T04:40:00Z
4,46086,182.0,7.7,9.3,1.79,10.81,7.15,259.0,1011.0,15.5,...,2.04,4.1616,7.9,12.12,1.76,3.0976,7.0,11.43,2023-01-01T05:10:00Z,46086_2023-01-01T05:10:00Z


In [38]:
from sagemaker.feature_store.feature_group import FeatureGroup

feature_group = FeatureGroup(
    name="buoy-wave-features-v2",
    sagemaker_session=sess,
)


In [39]:
from sagemaker.feature_store.feature_definition import FeatureDefinition
from sagemaker.feature_store.feature_definition import (
    FeatureTypeEnum as FeatureType
)

feature_definitions = [
    FeatureDefinition("record_id", FeatureType.STRING),
    FeatureDefinition("event_time", FeatureType.FRACTIONAL),
    FeatureDefinition("station_id", FeatureType.INTEGRAL),
]

for col in feature_cols + ["target_E_star"]:
    feature_definitions.append(
        FeatureDefinition(col, FeatureType.FRACTIONAL)
    )


In [40]:
feature_group.load_feature_definitions(
    data_frame=df_features
)


[FeatureDefinition(feature_name='station_id', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='wind_direction', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='wind_speed', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='wind_gust', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='wave_height', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='dominant_wave_period', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='average_wave_period', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='mean_wave_direction', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fract

In [41]:
feature_group.create(
    record_identifier_name="record_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=False,
    s3_uri=f"s3://{sess.default_bucket()}/feature-store/buoy/",
)


{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:318401170150:feature-group/buoy-wave-features-v2',
 'ResponseMetadata': {'RequestId': '43f23f5a-0c40-4a16-884d-2c2388cb5597',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '43f23f5a-0c40-4a16-884d-2c2388cb5597',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '98',
   'date': 'Tue, 10 Feb 2026 00:23:05 GMT'},
  'RetryAttempts': 0}}

In [52]:
feature_group.ingest(
    data_frame=df_features[
        ["record_id", "event_time", "station_id"]
        + feature_cols
        + ["target_E_star"]
    ],
    max_workers=4,
    wait=True,
)

IngestionManagerPandas(feature_group_name='buoy-wave-features-v2', feature_definitions={'station_id': {'FeatureName': 'station_id', 'FeatureType': 'String'}, 'wind_direction': {'FeatureName': 'wind_direction', 'FeatureType': 'Fractional'}, 'wind_speed': {'FeatureName': 'wind_speed', 'FeatureType': 'Fractional'}, 'wind_gust': {'FeatureName': 'wind_gust', 'FeatureType': 'Fractional'}, 'wave_height': {'FeatureName': 'wave_height', 'FeatureType': 'Fractional'}, 'dominant_wave_period': {'FeatureName': 'dominant_wave_period', 'FeatureType': 'Fractional'}, 'average_wave_period': {'FeatureName': 'average_wave_period', 'FeatureType': 'Fractional'}, 'mean_wave_direction': {'FeatureName': 'mean_wave_direction', 'FeatureType': 'Fractional'}, 'pressure': {'FeatureName': 'pressure', 'FeatureType': 'Fractional'}, 'air_temperature': {'FeatureName': 'air_temperature', 'FeatureType': 'Fractional'}, 'water_temperature': {'FeatureName': 'water_temperature', 'FeatureType': 'Fractional'}, 'dewpoint_temperat

In [53]:
feature_group.describe()["FeatureGroupStatus"]


'Created'

In [54]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>